# Speech Emotion Recognition (SER) Exploratory Data Analysis (EDA)

This notebook explores the pre-extracted acoustic features dataset to understand:
1. Data scale and feature statistics.
2. Distribution of sample counts per emotion class.
3. Feature correlation heatmaps.
4. Visual separation of emotion classes in 2D feature spaces.


In [ ]:
# ============================================================
# Library Imports
# ============================================================
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)


## Path Configuration

We locate the tabular data file `all_emotions.csv` at the project root or the `dataset/` subfolder.


In [ ]:
# ============================================================
# Path Configuration and Global Constants
# ============================================================
_base = os.getcwd()
_project_root = os.path.dirname(_base) if os.path.basename(_base).lower() == "training" else _base
ALL_EMOTIONS_CSV = os.path.normpath(os.path.join(_project_root, "dataset", "all_emotions.csv"))
if not os.path.isfile(ALL_EMOTIONS_CSV):
    fallback_csv = os.path.normpath(os.path.join(_project_root, "all_emotions.csv"))
    if os.path.isfile(fallback_csv):
        ALL_EMOTIONS_CSV = fallback_csv

print("Dataset path:", ALL_EMOTIONS_CSV)
print("Dataset file exists:", os.path.isfile(ALL_EMOTIONS_CSV))


## Load Dataset and Print Metadata

We load the CSV file using pandas and print basic info (column names, types, missing values).


In [ ]:
# ============================================================
# Load Dataset and Summarize Metadata
# ============================================================
df = pd.read_csv(ALL_EMOTIONS_CSV)
print("Dataset Info:")
df.info()


## Statistical Summaries

We print mean, standard deviation, and quartile ranges for each audio feature.


In [ ]:
# ============================================================
# Statistical Analysis of Features
# ============================================================
print("Statistical Summary of Acoustic Features:")
df.describe()


## Emotion Class Distribution

We inspect sample count distribution per target label class.


In [ ]:
# ============================================================
# Class Label Distribution Analysis
# ============================================================
plt.figure(figsize=(10, 5))
target_col = "label" if "label" in df.columns else "Label"

df[target_col].value_counts().plot(kind="bar", color="steelblue", edgecolor="black")
plt.xlabel("Emotion Class", fontsize=12)
plt.ylabel("Number of Samples", fontsize=12)
plt.title("Sample Distribution across Emotion Classes", fontsize=14)
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## Correlation Analysis: Mean Descriptors

We plot a heatmap of the correlation between mean values of main acoustic features.


In [ ]:
# ============================================================
# Correlation Matrix Analysis for Mean Descriptors
# ============================================================
cols_mean = [
    "F0_mean",
    "Energy_ mean",
    "ZCR_mean",
    "Spectral_centroid_mean",
    "Spectral_flux_mean",
    "MFCC_C4_mean",
    "Delta_MFCC_C4_mean",
]

available_cols = [c for c in cols_mean if c in df.columns]
if available_cols:
    plt.figure(figsize=(8, 6))
    corr_matrix_mean = df[available_cols].corr()
    sns.heatmap(corr_matrix_mean, annot=True, cmap="coolwarm", fmt=".2f", square=True)
    plt.title("Correlation Matrix of Mean Features", fontsize=14)
    plt.tight_layout()
    plt.show()


## Correlation Analysis: Standard Deviation Descriptors

We plot a correlation matrix for features measuring variance (standard deviation descriptors).


In [ ]:
# ============================================================
# Correlation Matrix Analysis for Standard Deviation Descriptors
# ============================================================
cols_std = [
    "F0_std",
    "Energy_ std",
    "ZCR_std",
    "Spectral_centroid_std",
    "MFCC_C4_std",
    "Delta_MFCC_C4_std",
]

available_cols_std = [c for c in cols_std if c in df.columns]
if available_cols_std:
    plt.figure(figsize=(8, 6))
    corr_matrix_std = df[available_cols_std].corr()
    sns.heatmap(corr_matrix_std, annot=True, cmap="coolwarm", fmt=".2f", square=True)
    plt.title("Correlation Matrix of Std Features", fontsize=14)
    plt.tight_layout()
    plt.show()


## Visual Separation Scatter plots: Mean Features


In [ ]:
# ============================================================
# Feature Scatter Plots: Spectral Centroid Mean vs ZCR Mean
# ============================================================
emotions = df[target_col].unique()
colors = plt.cm.Set3(np.linspace(0, 1, len(emotions)))

n_emotions = len(emotions)
ncols = 3
nrows = (n_emotions + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 10))
axes = axes.flatten()

for i, emotion in enumerate(emotions):
    subset = df[df[target_col] == emotion]
    axes[i].scatter(subset["Spectral_centroid_mean"], subset["ZCR_mean"], color=colors[i], alpha=0.5, edgecolor="none", s=15)
    axes[i].set_title(f"Emotion: {emotion}", fontsize=12)
    axes[i].set_xlabel("Spectral Centroid Mean", fontsize=10)
    axes[i].set_ylabel("ZCR Mean", fontsize=10)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Spectral Centroid Mean vs Zero Crossing Rate Mean", fontsize=16, y=0.98)
plt.tight_layout()
plt.show()


## Visual Separation Scatter plots: Standard Deviation Features


In [ ]:
# ============================================================
# Feature Scatter Plots: Spectral Centroid Std vs ZCR Std
# ============================================================
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 10))
axes = axes.flatten()

for i, emotion in enumerate(emotions):
    subset = df[df[target_col] == emotion]
    axes[i].scatter(subset["Spectral_centroid_std"], subset["ZCR_std"], color=colors[i], alpha=0.5, edgecolor="none", s=15)
    axes[i].set_title(f"Emotion: {emotion}", fontsize=12)
    axes[i].set_xlabel("Spectral Centroid Std", fontsize=10)
    axes[i].set_ylabel("ZCR Std", fontsize=10)

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle("Spectral Centroid Std vs Zero Crossing Rate Std", fontsize=16, y=0.98)
plt.tight_layout()
plt.show()
